# 서비스셋 임베딩 run — 반경 안 상위 결과 (Colab GPU)

서비스셋 쿼리마다 가정위치에서 반경(기본 2km) 안의 가맹점만 놓고 임베딩 모델 상위 결과를 만듭니다. 판정 pool과 채점에 씁니다.
계획은 `docs/SERVICE_ANNOTATION_GUIDELINE.md` 4절, 로컬 단계는 `scripts/build_service_pool.py`.

1. 로컬: `python scripts/build_service_pool.py export` → `colab_upload/service_eval/`의 `docs.csv`, `queries.csv`, `meta.json`을
   Drive `내 드라이브/store-search-ai/service_eval/`에 올림
2. 학습 모델(bge-m3 mj)이 Drive `store-search-ai/runs/finetune/bge_m3_ft_mj_20261014_0743/`에 없으면, 로컬 `models/bge_m3_ft_mj_20261014_0743/`
   폴더를 그 위치에 올림(아래 `MODELS`의 경로를 바꿔도 됨)
3. Colab: 런타임 GPU(T4 가능) → 위에서부터 실행 → 같은 폴더에 `<모델 이름>.csv`가 생김(이미 있으면 건너뜀)
4. 로컬: 그 파일들을 `benchmark/service_ko_v1/runs/`에 넣고 `python scripts/build_service_pool.py runs` → `pool --dry-run`

반경 안 후보 고르기·상위 k개(`# @cell runs`)는 로컬 `store_search_ai.data.service_pool`과 같은 규칙입니다
(`tests/test_service_dense_notebook.py`가 확인).

In [ ]:
import torch
assert torch.cuda.is_available(), "런타임 > 런타임 유형 변경 > GPU를 선택하세요"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DIR = Path("/content/drive/MyDrive/store-search-ai/service_eval")
# 이름(run 파일 이름 = 로컬 configs/models/*.yaml의 name) → 모델 경로. bge 계열은 query 지시문이 없다.
MODELS = {
    "bge_m3_ft_mj_20261014_0743": "/content/drive/MyDrive/store-search-ai/runs/finetune/bge_m3_ft_mj_20261014_0743",
    "bge_m3": "BAAI/bge-m3",
}
BATCH = 256

In [ ]:
# @cell runs
import numpy as np
import pandas as pd

EARTH_RADIUS_KM = 6371.0
RUN_COLUMNS = ["query_id", "doc_id", "rank", "score", "system"]


def haversine_km(lat, lon, lats, lons):
    a, b = np.radians(lat), np.radians(lon)
    la, lo = np.radians(lats), np.radians(lons)
    h = np.sin((la - a) / 2) ** 2 + np.cos(a) * np.cos(la) * np.sin((lo - b) / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(h))


def radius_runs(docs, queries, doc_emb, q_emb, radius_km, depth, system):
    """쿼리마다 반경 안 가맹점의 코사인 점수 상위 depth개. 동점은 docs 순서(로컬 top_k_run과 같은 규칙)."""

    lats, lons = docs["latitude"].to_numpy(float), docs["longitude"].to_numpy(float)
    doc_ids = docs["doc_id"].to_numpy()
    cache, frames = {}, []
    for i, (qid, lat, lon) in enumerate(zip(queries["query_id"], queries["latitude"], queries["longitude"])):
        key = (float(lat), float(lon))
        if key not in cache:
            cache[key] = np.flatnonzero(haversine_km(key[0], key[1], lats, lons) <= radius_km)
        cand = cache[key]
        if len(cand) == 0:
            continue
        scores = doc_emb[cand] @ q_emb[i]
        order = np.lexsort((cand, -scores))[:depth]
        frames.append(pd.DataFrame({"query_id": qid, "doc_id": doc_ids[cand[order]], "rank": np.arange(1, len(order) + 1),
                                    "score": scores[order], "system": system}))
    return pd.concat(frames, ignore_index=True)[RUN_COLUMNS]

In [ ]:
import json

docs = pd.read_csv(DIR / "docs.csv", encoding="utf-8-sig", dtype={"doc_id": str})
queries = pd.read_csv(DIR / "queries.csv", encoding="utf-8-sig", dtype={"query_id": str})
meta = json.loads((DIR / "meta.json").read_text(encoding="utf-8"))
RADIUS, DEPTH = meta["judge_radius_km"], meta["run_depth"]
print(f"가맹점 {len(docs)}개, 쿼리 {len(queries)}개, 반경 {RADIUS}km, 상위 {DEPTH}개")

In [ ]:
import gc

from sentence_transformers import SentenceTransformer

for name, path in MODELS.items():
    out = DIR / f"{name}.csv"
    if out.exists():
        print(f"{out.name}: 이미 있음 — 건너뜀")
        continue
    model = SentenceTransformer(path, device="cuda", model_kwargs={"torch_dtype": torch.float16})
    doc_emb = model.encode(docs["text"].fillna("").tolist(), batch_size=BATCH, normalize_embeddings=True,
                           show_progress_bar=True, convert_to_numpy=True).astype("float32")
    q_emb = model.encode(queries["query"].tolist(), batch_size=BATCH, normalize_embeddings=True,
                         convert_to_numpy=True).astype("float32")
    run = radius_runs(docs, queries, doc_emb, q_emb, RADIUS, DEPTH, name)
    run.to_csv(out, index=False, encoding="utf-8-sig", lineterminator="\n")
    print(f"[저장] {out} ({len(run)}행, 쿼리 {run['query_id'].nunique()}개)")
    del model, doc_emb
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
for name in MODELS:
    run = pd.read_csv(DIR / f"{name}.csv", encoding="utf-8-sig")
    print(name, len(run), "행 / 쿼리", run["query_id"].nunique(), "/ 쿼리당 평균", round(run.groupby("query_id").size().mean(), 1))
    print(run[run["rank"] <= 3].merge(queries, on="query_id").merge(docs, on="doc_id")[["query", "text", "rank"]].head(9).to_string(index=False))